# 02_graph_load.ipynb

This notebook connects to Neo4j, verifies the property graph load, and runs business queries for supply chain delay analysis.

# Business Question

“Which supply chain factors—across products, regions, and shipping modes—are driving late deliveries, and what predictive signals can be used to reduce delays?”

In [103]:
pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [104]:
!pip install yfiles-jupyter-graphs

In [105]:
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
import pandas as pd

uri = "bolt://localhost:7687"
username = "neo4j"
password = "Swara@1212"

driver = GraphDatabase.driver(uri, auth=(username, password))

print("Connected!")

Connected!


In [106]:
with driver.session() as session:
    nodes = session.run("MATCH (n) RETURN count(n) AS total").single()["total"]
    rels = session.run("MATCH ()-[r]->() RETURN count(r) AS total").single()["total"]

print("Total Nodes:", nodes)
print("Total Relationships:", rels)

Total Nodes: 65958
Total Relationships: 291503


In [107]:
# Relationship Type Counts
query = """
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(*) AS count
ORDER BY count DESC
"""
pd.DataFrame(driver.session().run(query).data())

,relationship,count
0,CONTAINS,159763
1,SHIPPED_TO,65752
2,USED_MODE,65752
3,IN_CATEGORY,118
4,IN_DEPARTMENT,118


In [108]:
# Label Counts
query = """
MATCH (n)
RETURN labels(n)[0] AS label, count(*) AS count
ORDER BY count DESC
"""
pd.DataFrame(driver.session().run(query).data())

,label,count
0,Order,65752
1,Product,118
2,Category,50
3,Region,23
4,Department,11
5,ShippingMode,4


In [109]:
#Relationship Counts
with driver.session() as session:
    print(session.run("MATCH ()-[r]->() RETURN count(r) AS total").single())

<Record total=291503>


In [110]:
# Node Count
with driver.session() as session:
    print(session.run("MATCH (n) RETURN count(n) AS total").single())

<Record total=65958>


In [111]:
def run_query(query):
    with driver.session() as session:
        return session.run(query).data()

### The graph has been loaded successfully. Orders are connected to products, regions, shipping modes, and departments through relationships.

#The graph has been loaded successfully. Orders are connected to products, regions, shipping modes, and departments through relationships.

In [112]:
run_query("""
MATCH (a)-[r]->(b)
RETURN a,r,b
LIMIT 10
""")

[{'a': {'price': 327.75,
   'product_id': 1360,
   'unique_visitors': 0.0,
   'name': 'Smart watch ',
   'views': 0.0},
  'r': ({'price': 327.75,
    'product_id': 1360,
    'unique_visitors': 0.0,
    'name': 'Smart watch ',
    'views': 0.0},
   'IN_CATEGORY',
   {'name': 'Sporting Goods'}),
  'b': {'name': 'Sporting Goods'}},
 {'a': {'price': 327.75,
   'product_id': 1360,
   'unique_visitors': 0.0,
   'name': 'Smart watch ',
   'views': 0.0},
  'r': ({'price': 327.75,
    'product_id': 1360,
    'unique_visitors': 0.0,
    'name': 'Smart watch ',
    'views': 0.0},
   'IN_DEPARTMENT',
   {'department_id': 2, 'name': 'Fitness'}),
  'b': {'department_id': 2, 'name': 'Fitness'}},
 {'a': {'price': 59.99000168,
   'product_id': 365,
   'unique_visitors': 2773.0,
   'name': 'Perfect Fitness Perfect Rip Deck',
   'views': 27878.0},
  'r': ({'price': 59.99000168,
    'product_id': 365,
    'unique_visitors': 2773.0,
    'name': 'Perfect Fitness Perfect Rip Deck',
    'views': 27878.0},
   

In [113]:
run_query("""
MATCH (o:Order)-[:SHIPPED_TO]->(r:Region)
RETURN r.name AS region,
       avg(o.late_flag) AS late_rate,
       count(o) AS total_orders
ORDER BY late_rate DESC
""")

[{'region': 'Central Africa',
  'late_rate': 0.5755395683453236,
  'total_orders': 556},
 {'region': 'East Africa',
  'late_rate': 0.567699836867863,
  'total_orders': 613},
 {'region': 'South of  USA ',
  'late_rate': 0.5598513011152413,
  'total_orders': 1345},
 {'region': 'West Asia',
  'late_rate': 0.5598417408506424,
  'total_orders': 2022},
 {'region': 'Eastern Europe',
  'late_rate': 0.559597523219815,
  'total_orders': 1292},
 {'region': 'South Asia',
  'late_rate': 0.5592203898050987,
  'total_orders': 3335},
 {'region': 'Western Europe',
  'late_rate': 0.5579420579420579,
  'total_orders': 10010},
 {'region': 'Southeast Asia',
  'late_rate': 0.5557851239669428,
  'total_orders': 4356},
 {'region': 'Central Asia',
  'late_rate': 0.5543478260869565,
  'total_orders': 184},
 {'region': 'US Center ',
  'late_rate': 0.5534883720930236,
  'total_orders': 1935},
 {'region': 'East of USA',
  'late_rate': 0.5492897115798534,
  'total_orders': 2323},
 {'region': 'Central America',
  'l

In [114]:
run_query("""
MATCH (o:Order)-[:CONTAINS]->(p:Product)-[:IN_DEPARTMENT]->(d:Department)
RETURN d.name AS department,
       avg(o.late_flag) AS late_rate,
       count(o) AS total_orders
ORDER BY late_rate DESC
""")

[{'department': 'Pet Shop',
  'late_rate': 0.5894308943089435,
  'total_orders': 492},
 {'department': 'Book Shop',
  'late_rate': 0.5654320987654323,
  'total_orders': 405},
 {'department': 'Health and Beauty ',
  'late_rate': 0.558011049723757,
  'total_orders': 362},
 {'department': 'Fitness',
  'late_rate': 0.5557357113903528,
  'total_orders': 2467},
 {'department': 'Outdoors',
  'late_rate': 0.5548534134465978,
  'total_orders': 9653},
 {'department': 'Technology',
  'late_rate': 0.5501706484641635,
  'total_orders': 1465},
 {'department': 'Fan Shop',
  'late_rate': 0.5486832486101443,
  'total_orders': 58819},
 {'department': 'Footwear',
  'late_rate': 0.5484038990996349,
  'total_orders': 13439},
 {'department': 'Apparel',
  'late_rate': 0.5483106965053849,
  'total_orders': 41378},
 {'department': 'Golf',
  'late_rate': 0.5467409508835532,
  'total_orders': 29257},
 {'department': 'Discs Shop',
  'late_rate': 0.5444225074037515,
  'total_orders': 2026}]

In [115]:
run_query("""
MATCH (o:Order)
RETURN o.shipping_mode AS shipping_mode,
       avg(o.late_flag) AS late_rate,
       count(o) AS total_orders
ORDER BY late_rate DESC
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `shipping_mode` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=10, offset=26>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 26, 'line': 3, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (o:Order)\nRETURN o.shipping_mode AS shipping_mode,\n       avg(o.late_flag) AS late_rate,\n       count(o) AS total_orders\nORDER BY late_rate DESC\n'


[{'shipping_mode': None,
  'late_rate': 0.5482418785740389,
  'total_orders': 65752}]

In [116]:
#Which product-region combinations have the most late deliveries
run_query("""
MATCH (o:Order {late_flag:1})-[:CONTAINS]->(pr:Product),
      (o)-[:SHIPPED_TO]->(r:Region)
RETURN pr.name AS product,
       r.name AS region,
       count(o) AS late_orders
ORDER BY late_orders DESC
LIMIT 10
""")

[{'product': 'Perfect Fitness Perfect Rip Deck',
  'region': 'Central America',
  'late_orders': 1840},
 {'product': 'Perfect Fitness Perfect Rip Deck',
  'region': 'Western Europe',
  'late_orders': 1725},
 {'product': "Nike Men's CJ Elite 2 TD Football Cleat",
  'region': 'Central America',
  'late_orders': 1650},
 {'product': "Nike Men's Dri-FIT Victory Golf Polo",
  'region': 'Central America',
  'late_orders': 1551},
 {'product': "O'Brien Men's Neoprene Life Vest",
  'region': 'Central America',
  'late_orders': 1549},
 {'product': "Nike Men's CJ Elite 2 TD Football Cleat",
  'region': 'Western Europe',
  'late_orders': 1527},
 {'product': "Nike Men's Dri-FIT Victory Golf Polo",
  'region': 'Western Europe',
  'late_orders': 1507},
 {'product': 'Field & Stream Sportsman 16 Gun Fire Safe',
  'region': 'Central America',
  'late_orders': 1360},
 {'product': 'Field & Stream Sportsman 16 Gun Fire Safe',
  'region': 'Western Europe',
  'late_orders': 1333},
 {'product': "O'Brien Men's 

### Interpretation

This query identifies product-region combinations most associated with late deliveries.

Products repeatedly delayed in certain regions may indicate logistics bottlenecks, inventory shortages, or regional fulfillment challenges.

### Business Value

Managers can prioritize stock planning, routing improvements, or alternate carriers for these combinations.

In [117]:
# This query links delayed orders to products and departments, helping identify which departments face higher late-delivery risk.
run_query("""
MATCH (o:Order {late_flag:1})-[:CONTAINS]->(p:Product)
-[:IN_DEPARTMENT]->(d:Department)
RETURN p.name AS product,
       d.name AS department,
       count(o) AS late_orders
ORDER BY late_orders DESC
LIMIT 10
""")

[{'product': 'Perfect Fitness Perfect Rip Deck',
  'department': 'Apparel',
  'late_orders': 11211},
 {'product': "Nike Men's CJ Elite 2 TD Football Cleat",
  'department': 'Apparel',
  'late_orders': 10247},
 {'product': "Nike Men's Dri-FIT Victory Golf Polo",
  'department': 'Golf',
  'late_orders': 9723},
 {'product': "O'Brien Men's Neoprene Life Vest",
  'department': 'Fan Shop',
  'late_orders': 9113},
 {'product': 'Field & Stream Sportsman 16 Gun Fire Safe',
  'department': 'Fan Shop',
  'late_orders': 8330},
 {'product': 'Pelican Sunstream 100 Kayak',
  'department': 'Fan Shop',
  'late_orders': 7549},
 {'product': "Diamondback Women's Serene Classic Comfort Bi",
  'department': 'Fan Shop',
  'late_orders': 6719},
 {'product': "Nike Men's Free 5.0+ Running Shoe",
  'department': 'Footwear',
  'late_orders': 6057},
 {'product': "Under Armour Girls' Toddler Spine Surge Runni",
  'department': 'Golf',
  'late_orders': 5418},
 {'product': 'Fighting video games',
  'department': 'Dis

### Interpretation

This query identifies products and departments that are most frequently associated with late deliveries.

The results show which specific products generate the highest number of delayed orders and which departments they belong to. If multiple delayed products come from the same department, it may indicate recurring operational issues such as high demand, stock shortages, complex handling requirements, or slower fulfillment processes.

### Business Value

These insights help supply chain managers prioritize high-risk departments, improve inventory planning, and optimize fulfillment strategies for products that are repeatedly linked to delays.

### Overall Findings

The graph analysis suggests that late deliveries are influenced by multiple connected factors:

- Product type
- Region
- Department
- Shipping mode

This demonstrates the value of graph databases for supply chain analytics because relationships across entities can be analyzed efficiently.